# Grilla del pipeline — Deliverable 2
**Phi-3.5-mini · 60 casos · 2 variantes × 2 modos · resultados en Google Drive**

Ejecuta las celdas **en orden, de arriba abajo**. No hay que editar ninguna.

La corrida completa son cuatro pasadas sobre los 60 casos. Si la sesión se corta, vuelve a
ejecutar la celda de la grilla: retoma donde iba y no repite lo ya medido.


## 1 · GPU y dependencias
Tarda unos dos minutos. Debe decir `cuda: True` y mostrar una **Tesla T4**.

Si no aparece la T4: menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* →
acelerador **T4 GPU**.


In [ ]:
!nvidia-smi -L
!pip -q install -U transformers accelerate bitsandbytes
import torch; print('cuda:', torch.cuda.is_available())

## 2 · Subir el paquete
Al ejecutar aparece un botón **Elegir archivos**. Sube `paquete_colab.zip` desde:

`C:\\Lucas\\Claude\\2026-2\\GenAI\\deliverable-1\\`

Espera a que termine de subir del todo antes de seguir.


In [ ]:
from google.colab import files
files.upload();

## 3 · Descomprimir y verificar
**Ésta es la celda que decide si se puede seguir.** Corre las cuatro autopruebas del
repositorio. Las cuatro tienen que pasar antes de gastar un segundo de GPU.

Debe terminar con `TODO EN ORDEN`.


In [ ]:
!unzip -o -q paquete_colab.zip -d proyecto
%cd /content/proyecto/scripts

import subprocess, sys

PRUEBAS = [
    ('verificador.py', ['verificador.py'],              '9/9 casos de control'),
    ('ficha.py',       ['ficha.py'],                    '60/60 desde la ficha'),
    ('pipeline.py',    ['pipeline.py'],                 '15/15 de parseo'),
    ('runner_d2.py',   ['runner_d2.py', '--pruebas'],   '6/6 del encadenado'),
]

todo_ok = True
for nombre, args, que in PRUEBAS:
    r = subprocess.run([sys.executable] + args, capture_output=True, text=True)
    ultima = (r.stdout.strip().splitlines() or ['(sin salida)'])[-1]
    estado = 'OK  ' if r.returncode == 0 else 'FALLA'
    print(f'[{estado}] {nombre:16} {ultima}')
    if r.returncode != 0:
        todo_ok = False
        print(r.stdout[-1500:])
        print(r.stderr[-800:])

print()
print('TODO EN ORDEN' if todo_ok else '*** NO SIGAS: alguna prueba falló ***')

## 4 · Conectar Google Drive
**Este es el paso que evita perder el trabajo.** Son cuatro corridas seguidas, así que esta
vez sí conviene montarlo. Te va a pedir autorización: acéptala con tu cuenta de Google.

Desde acá, todo lo que se mida se escribe en `MyDrive/genai_resultados/` en vez del disco
temporal de Colab.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/genai_resultados
!rm -rf /content/proyecto/resultados
!ln -s /content/drive/MyDrive/genai_resultados /content/proyecto/resultados

print('\nresultados guardados en Drive. Lo que ya hay:')
!ls -la /content/drive/MyDrive/genai_resultados/

## 5 · Modelo
Phi-3.5-mini, 3,8 mil millones de parámetros, cuantizado a 4 bits. Elegido en el diseño por
tener el mejor acierto de decisión fuera de su categoría de colapso, y por ser el más chico
de los tres candidatos del Deliverable 1.


In [ ]:
MODELO = 'microsoft/Phi-3.5-mini-instruct'
print('modelo elegido:', MODELO)

## 6 · La grilla
Cuatro corridas de 60 casos.

| variante | modo | qué mide |
|---|---|---|
| `puro` | `encadenado` | el sistema: el modelo hace los tres pasos |
| `puro` | `oraculo` | la competencia de cada paso por separado |
| `retrieval` | `encadenado` | el sistema con la ficha armada por código |
| `retrieval` | `oraculo` | el paso 3 solo, con ficha perfecta |

Calcula entre 20 y 30 minutos incluyendo la carga del modelo. **Si algo se corta, vuelve a
correr esta misma celda**: retoma donde iba.


In [ ]:
import subprocess, sys, os, time

os.chdir('/content/proyecto/scripts')
GRILLA = [('puro', 'encadenado'), ('puro', 'oraculo'),
          ('retrieval', 'encadenado'), ('retrieval', 'oraculo')]

t0 = time.time()
for var, modo in GRILLA:
    print('=' * 72); print(f'{MODELO}  ·  variante {var}  ·  modo {modo}'); print('=' * 72)
    cmd = [sys.executable, 'runner_d2.py', '--modelo', MODELO,
           '--variante', var, '--modo', modo]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode == 0:
        print('\n'.join(r.stdout.strip().splitlines()[-16:]))
    else:
        print('FALLO:')
        print('\n'.join(r.stderr.strip().splitlines()[-10:]))
        break
    print()

print(f'grilla completa en {(time.time() - t0) / 60:.1f} minutos')

## 7 · Tabla comparativa
El baseline arriba, las cuatro corridas del pipeline abajo. Ésta es la tabla que va al
documento.


In [ ]:
import json, glob, os

print('{:34} {:>9} {:>8} {:>8} {:>9} {:>8}'.format(
    'condicion', 'decision', 'regla', 'ambas', 'tok/caso', 's/caso'))
print('-' * 82)

# baseline del D1
f = '/content/proyecto/baseline/Phi-3.5-mini-instruct__few_shot__prosa.raw.jsonl'
if os.path.exists(f):
    r = [json.loads(l) for l in open(f, encoding='utf-8') if l.strip()]
    p = lambda k: 100 * sum(x[k] for x in r) / len(r)
    print('{:34} {:8.1f}% {:7.1f}% {:7.1f}% {:9,.0f} {:8.1f}'.format(
        'baseline few-shot (1 llamada)', p('acierto_decision'), p('acierto_regla'),
        p('acierto_conjunto'),
        sum(x['tokens_entrada'] for x in r) / len(r),
        sum(x['segundos'] for x in r) / len(r)))
else:
    print('  (falta el baseline: regenera el paquete, deberia traerlo en baseline/)')

for f in sorted(glob.glob('/content/proyecto/resultados/pipeline__*.raw.jsonl')):
    r = [json.loads(l) for l in open(f, encoding='utf-8') if l.strip()]
    if not r:
        continue
    p = lambda k: 100 * sum(x[k] for x in r) / len(r)
    nom = f'{r[0]["variante"]} / {r[0]["modo"]}'
    print('{:34} {:8.1f}% {:7.1f}% {:7.1f}% {:9,.0f} {:8.1f}'.format(
        nom, p('acierto_decision'), p('acierto_regla'), p('acierto_conjunto'),
        sum(x['tokens_total'] for x in r) / len(r),
        sum(x['segundos_total'] for x in r) / len(r)))

## 8 · Atribución por paso
Dónde se rompe el sistema. Esta tabla es la que sostiene el criterio de lectura de los
límites de la rúbrica.


In [ ]:
import json, glob, os

for f in sorted(glob.glob('/content/proyecto/resultados/pipeline__*.raw.jsonl')):
    r = [json.loads(l) for l in open(f, encoding='utf-8') if l.strip()]
    if not r:
        continue
    n = len(r)
    print('=' * 72)
    print(f'variante {r[0]["variante"]}  ·  modo {r[0]["modo"]}   (n={n})')
    print('=' * 72)
    print(f'  paso 1  ramo {100*sum(x["paso1"]["acierto_ramo"] for x in r)/n:5.1f}%'
          f'   periodo {100*sum(x["paso1"]["acierto_periodo"] for x in r)/n:5.1f}%')
    for c in r[0]['paso2']['acierto']:
        print(f'  paso 2  {c:24} {100*sum(x["paso2"]["acierto"][c] for x in r)/n:5.1f}%')
    print(f'  paso 3  decision {100*sum(x["acierto_decision"] for x in r)/n:5.1f}%'
          f'   regla {100*sum(x["acierto_regla"] for x in r)/n:5.1f}%')
    print(f'  CONJUNTO {100*sum(x["acierto_conjunto"] for x in r)/n:5.1f}%')
    print()

## 9 · Descargar
Los resultados ya están en tu Drive. Esto te los baja también al computador, para
comitearlos al repositorio.


In [ ]:
import shutil
shutil.make_archive('/content/resultados_d2', 'zip', '/content/drive/MyDrive/genai_resultados')
from google.colab import files
files.download('/content/resultados_d2.zip')